# Anisotropy of magnetic susceptibility

The `apsg.ams` module handles anisotropy of magnetic susceptibility (AMS): the susceptibility of a rock depends on the direction of the applied field, and the susceptibility tensor of a specimen is a second-rank symmetric tensor whose principal axes $k_1 \ge k_2 \ge k_3$ reflect the fabric of the magnetic grains. The module is not imported by `from apsg import *`, so we import it separately.

The tutorial has two parts:

1. **Data and statistics**: specimens and sites with core orientation and bedding, coordinate systems, parameters of Jelinek (1981), the mean tensor of a site with the confidence regions of its axes (Jelinek, 1978) and tables.
2. **Grain models**: how a population of grains turns in a flow and what susceptibility tensor it gives, the March/Willis model of preferred orientation (Willis, 1977; Jezek and Hrouda, 2002).

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np

from apsg import *
from apsg import data
from apsg.ams import *

## The data

The examples use an **artificial** data set, `ams_two_sites`: 26 specimens of two sites, made with the grain models of the second part. Site `AM1` has oblate (flattened) paramagnetic grains in flattening strain, site `AM2` has prolate magnetite grains in a transpression. Every specimen has its core orientation (azimuth and hade) and the bedding of the site, the way they would come from an AGICO Kappabridge file and field notes. `data.load` gives a dictionary of `AMSSite` objects.

In [ ]:
sites = data.load('ams_two_sites')
sites

## Specimens

An `AMSSpecimen` holds a symmetric susceptibility tensor in the coordinates of the core, the bulk susceptibility `bulk` (AGICO files give a normalised tensor and the mean susceptibility separately), the standard error of the tensor fit, the orientation of the core and the bedding. `params()` gives the parameters of Jelinek (1981): the lineation $L = k_1/k_2$, the foliation $F = k_2/k_3$, the degree $P = k_1/k_3$, the corrected degree $P_j$ and the shape $T$ (+1 for an oblate, -1 for a prolate ellipsoid).

In [ ]:
s = sites['AM2']['AM2-1']
print(s)
s.params()

The tensor is rotated to other coordinate systems by `tensor_in()` and `ellipsoid()`: `'specimen'` (the core), `'geographic'` (x north, y east, z down) and `'tectonic'` (geographic, with the bedding restored to horizontal). The orientation of the core uses the convention of the paleomagnetic module (`azimuth` and `hade` of the core axis); the bedding is given as dip direction and dip.

In [ ]:
for coords in ('specimen', 'geographic', 'tectonic'):
    k = s.ellipsoid(coords).eigenlins()
    print(f'{coords:10}', ', '.join(str(lin) for lin in k))

## Sites

An `AMSSite` is a list of specimens. `table()` gives the parameters of every specimen with the directions of the principal axes in the chosen coordinates, as a `DataFrame`.

In [ ]:
am1, am2 = sites['AM1'], sites['AM2']
am2.table('tectonic').round(3).head()

`tensors()` returns the normalised tensors (each divided by its mean) as an `EllipsoidSet`, which is what the statistics and plots of APSG work with. `AMSStereoPlot` shows the principal axes of the specimens (squares $k_1$, triangles $k_2$, circles $k_3$) in geographic and in tectonic coordinates, the latter with the bedding restored to horizontal. The foliation pole $k_3$ of the oblate site would be vertical there; the prolate site `AM2` has a horizontal lineation.

In [ ]:
AMSStereoPlot(am2, coords='both', ellipses=None).show()

### Mean tensor and confidence regions

`mean()` follows Jelinek (1978), who builds on Hext (1963): the mean of the normalised tensors, the confidence intervals of its eigenvalues and the confidence ellipse of each principal axis. The same regions are drawn by `StereoNet.confidence()` for a set of tensors. The `anisotropy_test` entry is Jelinek's test that the mean tensor is isotropic.

In [ ]:
m = am2.mean('tectonic')
m['eigenvalues'], m['shape'], m['anisotropy_test']

In [ ]:
es = am2.tensors('tectonic')
AMSStereoPlot(am2).show()

### Bootstrap

The linear perturbation method assumes small, normal errors. The **bootstrap** (Constable and Tauxe, 1990) resamples the normalised tensors of the site with replacement and collects the eigenvectors and eigenvalues of each resampled mean, so it makes no such assumption and shows the real spread of the axes. `bootstrap()` returns the bootstrapped eigenvalues and eigenvectors, the regions containing 95 % of the bootstrapped axes (`ellipses`, the same layout as in `mean()`) and the percentile intervals of the eigenvalues and of the parameters $P_j$, $T$. Note that for a small number of specimens the percentile regions cover the true axis less often than their level, so they are a measure of spread more than a guaranteed confidence region.

In [ ]:
boot = am2.bootstrap('tectonic', n_resamples=2000, rng=1)
print(boot)
print(boot.intervals.round(4))
boot.parameter_intervals()['Pj'], boot.parameter_intervals()['T']

`AMSStereoPlot(ellipses='bootstrap')` draws the specimen axes with the bootstrap regions of the mean tensor (`StereoNet.confidence(method='bootstrap_tensor')`). The eigenvalues of `AM2` are well separated, so the regions are close to the linearised ones.

In [ ]:
AMSStereoPlot(am2, ellipses='bootstrap', n=2000).show()

`BootstrapPlot` shows the bootstrapped axes with their regions and the histograms of the three eigenvalues. Their modes distinguish the shape classes of Constable and Tauxe (1990): well separated eigenvalues mean a triaxial fabric, two overlapping ones a prolate or oblate fabric.

In [ ]:
BootstrapPlot(am2, n=2000, rng=1).show()

The bootstrap also gives the uncertainty of **one specimen** from the standard error of the instrument (`std_error`): normal noise is added to the six tensor components and the simulated tensors are normalised and rotated to the chosen coordinates.

In [ ]:
spec_boot = am2['AM2-1'].bootstrap('tectonic', rng=2)
[tuple(round(g, 2) for g in e['gamma']) for e in spec_boot.ellipses]

### Comparing sites

`compare()` tests whether two sites have the same mean tensor. The statistic is Hotelling's $T^2$ of the difference of the two mean tensors, expressed by the five independent components of the deviatoric tensors, with the pooled covariance matrix (the two-sample extension of the isotropy test). The $p$-value comes from the $F$ distribution (`method='hotelling'`, for normal components with a common covariance) or from random reassignment of the specimens to the sites (`method='permutation'`), which needs no assumption. The sites `AM1` and `AM2` have different fabrics, while two halves of the same site do not differ.

In [ ]:
print(am1.compare(am2, 'tectonic'))
print(am1.compare(am2, 'tectonic', method='permutation', rng=0))
AMSSite(am2[::2]).compare(AMSSite(am2[1::2]), 'tectonic')

### Shape and degree of anisotropy

The two sites differ in the type of fabric. `PjTPlot` is the Jelinek diagram of the corrected degree $P_j$ against the shape $T$ of every specimen, one colour per site: `AM1` is oblate (foliated), `AM2` is prolate (lineated). `PTPlot` uses $P = k_1/k_3$ instead.

In [ ]:
PjTPlot(sites, coords='tectonic', mean=True).show()

`LFPlot` is the Flinn-type diagram of the lineation $L = k_1/k_2$ against the foliation $F = k_2/k_3$, with the line $L = F$ separating prolate from oblate fabrics.

In [ ]:
LFPlot(sites, coords='tectonic').show()

### Reading and writing tables

`to_csv()` writes a site as one table, the columns are the six independent components of the tensor in the order of AGICO files (`k11 k22 k33 k12 k23 k13`), the bulk susceptibility, the orientation of the core and the bedding. `read_csv()` and `from_dataframe()` read such tables, with several sites distinguished by the column `site`, and accept the usual alternative column names (`specimen`, `km`, `sigma`, `dip_direction`, ...). Other columns are kept in `meta`.

In [ ]:
print(am1.to_csv()[:450])
back = read_csv(io.StringIO(am1.to_csv()))
back

### Reading instrument files

Reports of the SUFAR program and tables exported by Anisoft are read by `read_ams()`. A report (`.asc`) holds one page per specimen with the core orientation, the mean susceptibility, the standard errors and the normed tensor in specimen coordinates; a table (`.txt`) has the tensors in geographic coordinates, or, in the shortest export, only the principal directions and the ratios $L$ and $F$, from which the tensor is rebuilt. Binary `.ams` files have to be exported to text in Anisoft first. `AMSSite.split()` groups the specimens of one table into sites by their names, for example `GC13_4` into site `GC13`.

```python
site = read_ams('HK10C.ASC')
sites = read_ams('M162-AMS.txt', site=lambda name: name.split('_')[0])
```

## Grain models

The susceptibility of a rock is the average of the susceptibilities of its grains. For **uniaxial** grains (prolate: the long axis is the unique axis, oblate: the pole of the flat grain) with transverse susceptibility $K$ and anisotropy $\Delta = K_\text{axis} - K_\text{transverse}$ it is

$$ \mathbf{k} = K\,\mathbf{I} + \Delta\,\mathbf{T}, $$

where $\mathbf{T}$ is the orientation tensor of the grain axes (trace 1). $\Delta > 0$ for prolate and $\Delta < 0$ for oblate grains. `ams_from_orientation()` evaluates this for an orientation tensor, a distribution or a set of axes, `ams_from_grains()` for a mixture of several populations.

The shape of a grain controls how it turns in a flow. A rigid grain that does not interact with others rotates in a homogeneous flow like a material line carried by the **pseudo-velocity-gradient** $K\mathbf{E} + \mathbf{W}$, where $\mathbf{E}$ and $\mathbf{W}$ are the rate of deformation and the spin of the flow and $K$ the kinematic coefficient of the grain (Willis, 1977). `grain_K()` gives it, for spheroids Jeffery's $\pm (r^2 - 1)/(r^2 + 1)$ of the axial ratio $r$:

In [ ]:
for kind, ratio in (('prolate', 1.5), ('prolate', 3), ('prolate', 10), ('oblate', 3), ('cylinder', 3 * np.sqrt(3))):
    print(f'{kind:9} {ratio:5.2f}  K = {grain_K(kind, ratio):6.3f}')

A needle ($K = 1$) is a passive line and follows the deformation, a sphere ($K = 0$) only rotates with the spin, and a plate has $K = -1$. The shape-controlled anisotropy of a grain follows from its intrinsic susceptibility and the demagnetising factors (`osborn_factors()`), for example magnetite with $k = 16.6$ SI:

In [ ]:
print(osborn_factors(1.2))
grain_anisotropy(16.6, 1.2)

### Fabric of grains in a flow

`VelocityGradient3.pseudo()` builds the pseudo-velocity-gradient of a grain in a given flow and `defgrad()` integrates it to the pseudo-displacement $\mathbf{D}_W$. Grains that start in all orientations with the same probability then have the density of March (1932), which is exactly an **angular central Gaussian distribution**, available as `ACG.from_displacement()`: lines (needles) for prolate grains, `kind='poles'` for the poles of plates. Its orientation tensor is known exactly, `ortensor()`.

As an example take a transpression: simple shear with shortening across the shear zone and extension along the vertical. We follow prolate grains of axial ratio 3 and make their AMS tensor with $K = 0.9$ and $\Delta = 0.3$.

In [ ]:
alpha = np.radians(10)
L = velgrad([[0, 1, 0], [0, -np.tan(alpha), 0], [0, 0, np.tan(alpha)]])
K = grain_K('prolate', 3)
Lg = L.pseudo(K)

for gamma in (1, 3, 6):
    D = Lg.defgrad(gamma)
    fabric = ACG.from_displacement(D)
    k = ams_from_orientation(fabric, K=0.9, delta=0.3)
    print(f'gamma = {gamma}: grain lineation {fabric.axes[0]}, AMS k1 {k.eigenlins()[0]}, T = {k.T:.2f}')

The grains first align with the flow direction and, with growing strain, the extension in the vertical takes over: the lineation of the grain fabric (and the AMS) **switches** from horizontal to vertical. The switch comes earlier for shorter grains, because they turn faster in the shear (Jezek, 2026).

In [ ]:
gamma = np.linspace(0.05, 12, 240)
fig, ax = plt.subplots(figsize=(6, 3.6), layout='constrained')
for r in (1.5, 3, 10):
    Lr = L.pseudo(grain_K('prolate', r))
    plunge = [abs(ACG.from_displacement(Lr.defgrad(g)).axes[0].geo[1]) for g in gamma]
    ax.plot(gamma, plunge, label=f'r = {r}')
ax.set_xlabel('shear strain $\\gamma$')
ax.set_ylabel('plunge of the grain lineation')
ax.axhline(45, color='gray', lw=0.5)
ax.legend();

The distribution can also be sampled (`rvs()`), plotted with `StereoNet.density()` and, for real data, fitted (`linset.fit('acg')`), which estimates the scatter matrix of the pseudo-strain from measured axes.

In [ ]:
fabric = ACG.from_displacement(Lg.defgrad(3))
axes = fabric.rvs(500, rng=np.random.default_rng(1))
net = StereoNet(title='grain axes, gamma = 3')
net.contour(axes)
net.point(axes, alpha=0.3)
net.show()

### Flow models

`AMSModel` wraps this into a model of a rock: grain populations (`GrainPopulation`, with the shape, the susceptibilities and a weight) carried by a flow. `evolve()` gives the AMS at the chosen times as an `AMSEvolution`, with the table of the parameters and the orientation of $k_1$, and `switch()` finds the strain at which the lineation switches. `ModelPlot` shows the evolution of $P_j$, $T$ and the plunges.

In [ ]:
times = np.linspace(0, 12, 241)
models = {f'r = {r}': AMSModel(L, GrainPopulation('prolate', r)).evolve(times) for r in (1.5, 3, 10)}
print({name: ev.switch() for name, ev in models.items()})
ModelPlot(models, xlabel='shear strain').show()

The shape of grains in the flow also decides the type of the fabric: the same flow with oblate grains gives a foliation. Populations can be mixed, with their weights.

In [ ]:
mix = AMSModel(L, [GrainPopulation('prolate', 3, delta=0.2), GrainPopulation('oblate', 4, delta=-0.1, weight=0.5)])
mix.evolve(times).table().iloc[[40, 120, 240]].round(3)

### Jeffery orbits

The model above treats spheroids. `jeffery_evolution()` integrates the rotation of rigid ellipsoids with **three different semi-axes** in the flow, after Jeffery (1922), for a sample of grains with random initial orientations. For a spheroid it gives the same fabric as the model above, up to the sampling error; a triaxial grain does not return to its orientation after a fixed strain, so its fabric differs from the spheroids' in the details.

In [ ]:
jeff = jeffery_evolution(L, [3, 1, 1], [1.3, 1, 1], times, n=600, rng=1)
tri = jeffery_evolution(L, [3, 2, 1], [1.3, 1.1, 1.0], times, n=600, rng=1)
ModelPlot({'spheroid': jeff, 'triaxial': tri}, xlabel='shear strain').show()

### From the AMS ellipsoid to the grain orientation

For a rock with one kind of uniaxial grains of known anisotropy $P_c$ (biotite, for example) the AMS ellipsoid can be converted into the orientation tensor of the grains (Hrouda and Schulmann, 1990). The result has the eigenvectors of the AMS ellipsoid, and the eigenvalues of an orientation tensor (trace 1); a tensor that no distribution of such grains can give raises an error. The method is not for magnetite, whose grains are triaxial.

In [ ]:
k = ams_from_orientation(ACG.from_displacement(Lg.defgrad(3)), K=0.9, delta=0.3)
O = orientation_tensor_from_ams(k, Pc=1.2 / 0.9, kind='prolate')
O.eigenvalues(), ACG.from_displacement(Lg.defgrad(3)).ortensor().eigenvalues()

## References

Constable, C., Tauxe, L. (1990). The bootstrap for magnetic susceptibility tensors. Journal of Geophysical Research 95(B6), 8383-8395.

Hrouda, F., Schulmann, K. (1990). Conversion of the magnetic susceptibility tensor into the orientation tensor in some rocks. Physics of the Earth and Planetary Interiors 63, 71-77.

Jeffery, G.B. (1922). The motion of ellipsoidal particles immersed in a viscous fluid. Proceedings of the Royal Society of London A 102, 161-179.

Hext, G.R. (1963). The estimation of second-order tensors, with related tests and designs. Biometrika 50, 353-373.

Jelinek, V. (1978). Statistical processing of anisotropy of magnetic susceptibility measured on groups of specimens. Studia Geophysica et Geodaetica 22, 50-62.

Jelinek, V. (1981). Characterization of the magnetic fabric of rocks. Tectonophysics 79, T63-T67.

Jezek, J. (2026). Lineation of the anisotropy of magnetic susceptibility in the wrench-dominated transpression. Studia Geophysica et Geodaetica 70, 5.

Jezek, J., Hrouda, F. (2002). Software for modeling the magnetic fabric of rocks in a deformation. Computers & Geosciences 28, 1061-1068.

March, A. (1932). Mathematische Theorie der Regelung nach der Korngestalt bei affiner Deformation. Zeitschrift fuer Kristallographie 81, 285-297.

Willis, D.G. (1977). A kinematic model of preferred orientation. GSA Bulletin 88, 883-894.